# FightWatch — pilot nhỏ với nhãn reviewed riêng

Tiêu chí đã xác nhận: động tác đấm/đá/vật/xô ngã có hướng vào người khác, kể cả mô phỏng an toàn. Không báo chỉ vì đứng gần, tranh cãi, can ngăn, vung tay một mình hoặc chỉ nằm trên đất sau sự việc.

Bộ thử: 62 cửa sổ từ 8 video, AI hỗ trợ rà khi ẩn nhãn/score; 11 cửa sổ chưa rõ không đưa vào bản curated. Còn 28 clip train từ 4 video và 23 clip validation từ 4 video khác. Không sửa annotation gốc, không trộn video nguồn qua split. Đây là **feasibility development rất nhỏ, không phải nhãn chuẩn do người độc lập xác minh, không phải độ chính xác camera thật**.

Notebook dùng code từ repo, có cell pull. Nếu đã có kết quả hoàn tất thì chỉ đọc lại, không train lại, không cần GPU để đọc báo cáo. Chỉ chọn L4 nếu chủ động chạy một output mới; budget tiến trình train 8 phút. Không tự ghi đè model local.

In [ ]:
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
import os, sys, json, shutil, subprocess, zipfile, re
ROOT = Path('/content/drive/MyDrive/SENTINEL')
REPO = ROOT / 'repo/anomaly-behavior-detector-system'
LOCAL = Path('/content/fightwatch-reviewed-code')
SOURCE = ROOT / 'artifacts/fight_expand_v1'
REVIEW = ROOT / 'artifacts/fight_curated_review_v1'
OUTPUT = ROOT / 'artifacts/fight_curated_pilot_v1'
DATA = Path('/content/fight_expand_data/UBI_FIGHTS')
CACHE = Path('/content/fight_curated_cache')
JUDGMENTS = REVIEW / 'judgments_frozen.json'
URL = 'https://github.com/TranDuon/anomaly-behavior-detector-system.git'
def completed():
    path = OUTPUT / 'comparison.json'
    return path.exists() and json.loads(path.read_text()).get('status') == 'completed'
print('Existing completed report:', completed())
assert (SOURCE / 'cached_manifest.json').exists() and (REVIEW / 'private_mapping.json').exists(), 'Thiếu artifact nguồn/review trên Drive'


## Pull code mới nhất
Bản repo trên Drive là bản đồng bộ từ local → GitHub: reset tracked changes theo quy trình đã thống nhất, không xóa dataset/artifact. Không đổi revision ghi trong kết quả cũ.

In [ ]:
REPO.parent.mkdir(parents=True, exist_ok=True)
LOCAL.mkdir(parents=True, exist_ok=True)
if (REPO / '.git').exists():
    remote = subprocess.check_output(['git','-C',str(REPO),'remote','get-url','origin'],text=True,timeout=60).strip()
    assert remote.removesuffix('.git') == URL.removesuffix('.git'), 'Sai remote'
    subprocess.run(['git','-C',str(REPO),'fetch','origin','main'],check=True,timeout=180)
    subprocess.run(['git','-C',str(REPO),'reset','--hard','origin/main'],check=True,timeout=120)
else:
    subprocess.run(['git','clone','--branch','main','--single-branch',URL,str(REPO)],check=True,timeout=240)
for raw in subprocess.check_output(['git','-C',str(REPO),'ls-files','-z']).split(b'\0'):
    if not raw:
        continue
    relative = Path(os.fsdecode(raw))
    if relative.name == '.env':
        continue
    destination = LOCAL / relative
    destination.parent.mkdir(parents=True,exist_ok=True)
    shutil.copy2(REPO / relative,destination)
os.chdir(LOCAL)
revision = subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()
from scripts.colab_process import run_logged
print('Code pulled:', revision)


## Môi trường và chỉ 8 video cần thiết
Không tải lại archive 8,1GB. Nếu đã có report, bỏ qua cài thư viện và giải nén. Nếu cần chạy mới, chọn L4 trước; notebook chỉ giải nén các video nằm trong mapping reviewed, không giải nén toàn dataset. Ngắt cell sẽ dừng tiến trình con thay vì để train nền.

In [ ]:
if completed():
    print('Đã có kết quả; bỏ qua setup/GPU/staging.')
else:
    import torch
    assert torch.cuda.is_available(), 'Chọn GPU L4 nếu muốn chạy pilot mới'
    OUTPUT.mkdir(parents=True,exist_ok=True)
    run_logged([sys.executable,'-m','pip','install','-q','-r','requirements-colab.txt'],OUTPUT / 'setup_log.txt',480,revision)
    run_logged([sys.executable,'-m','pytest','tests/test_fight_curated.py','tests/test_fight_pairs.py','tests/test_colab_process.py','-q','-p','no:cacheprovider'],OUTPUT / 'setup_log.txt',180,revision)
    assert JUDGMENTS.exists(), 'Thiếu judgments đã chốt; không tự tạo nhãn từ model score'
    mapping = json.loads((REVIEW / 'private_mapping.json').read_text())
    archive_path = ROOT / 'datasets/fight/UBI_FIGHTS.zip'
    assert archive_path.stat().st_size == 8_117_112_419, 'Archive gốc không khớp'
    with zipfile.ZipFile(archive_path) as archive:
        names = archive.namelist()
        for key in sorted({row['key'] for row in mapping['rows']}):
            assert re.fullmatch(r'[A-Za-z0-9_-]+',key)
            member = next(name for name in names if '/videos/' in '/' + name and name.endswith('/' + key + '.mp4'))
            destination = DATA / 'videos' / ('fight' if key.startswith('F_') else 'normal') / (key + '.mp4')
            assert destination.resolve().is_relative_to(DATA.resolve())
            destination.parent.mkdir(parents=True,exist_ok=True)
            if not destination.exists() or destination.stat().st_size != archive.getinfo(member).file_size:
                temporary = destination.with_suffix('.partial')
                with archive.open(member) as source, temporary.open('wb') as target:
                    shutil.copyfileobj(source,target)
                temporary.replace(destination)
    CACHE.mkdir(parents=True,exist_ok=True)
    print('Staged',len({row['video'] for row in mapping['rows']}),'source videos')


## Hai nhánh crop, giữ cùng thời gian và nhãn
8 phút tối đa, cùng pretrained initialization/seed/optimizer và cùng clip. Chỉ so tubes với pairs+context ở 16 frame/8fps. Chưa thử native 13 frame/5fps trên nhãn curated này vì thời lượng cửa sổ review khác; không âm thầm kéo dài cửa sổ và dùng lại nhãn.

Nhánh 2×2 trên nhãn gốc đã dừng trước train vì audit báo nhiều nghi vấn. Không dùng headline của 568 clip cũ để so trực tiếp với 23 clip curated. Không dùng official test. Không tự nối thêm giờ hoặc phê duyệt demo.

In [ ]:
if completed():
    print('Report completed; không train lại:', OUTPUT / 'comparison.json')
else:
    (OUTPUT / 'code_revision.txt').write_text(revision)
    command = [sys.executable,'-u','-m','scripts.fight_curated_pilot','--source-run',str(SOURCE),'--review-dir',str(REVIEW),'--judgments',str(JUDGMENTS),'--output-dir',str(OUTPUT),'--cache-dir',str(CACHE),'--budget-minutes','8']
    run_logged(command,OUTPUT / 'run_log.txt',510,revision)


## Kết quả và giới hạn
Reference expanded là model cũ train nhiều clip với nhãn gốc, không cùng budget train với hai nhánh nhỏ. Các threshold đều được chọn trên development; không coi precision/recall ở đây là cam kết cho camera thật. AP/AUC không phải accuracy. Xem cả false-positive rate và in-video AUC, không chỉ chọn số đẹp.

In [ ]:
import pandas as pd
from IPython.display import display, Image
report = json.loads((OUTPUT / 'comparison.json').read_text())
records = []
reference = report['expanded_reference_on_curated_validation']
records.append({'model':'expanded_reference',**reference['validation'],'recall_at_precision75':reference['best_recall_at_precision75']})
for name, result in report['variants'].items():
    records.append({'model':name,**result['validation'],'recall_at_precision75':result['best_recall_at_precision75']})
table = pd.DataFrame(records).set_index('model')
display(table[['ap','auc','precision','recall','false_positive_rate','in_video_auc','recall_at_precision75']].round(4))
print('Counts:',report['counts'])
print('Scope:',report['scope'])
print('Human verified: False | Demo approved:',report['approved_for_demo'])
for name in report['variants']:
    image_path = OUTPUT / name / 'crowd_crops.jpg'
    if image_path.exists():
        print(name,': source → actual model inputs')
        display(Image(filename=str(image_path),width=900))
print('Saved:',OUTPUT)


## Tải một gói CPU thử nghiệm (tùy chọn)
Chủ động chọn variant, không tự chọn theo AP và không tự thay model local. Để ZIP trong Downloads rồi chạy `run_local.cmd`, bật chế độ cho phép checkpoint pilot khi thử. Không dùng bộ tiny này để khẳng định demo đã đạt.

In [ ]:
VARIANT = 'tubes'
assert VARIANT in {'tubes','pairs'}
from google.colab import files
bundle = OUTPUT / VARIANT / 'sentinel_fight_only_runtime.zip'
assert bundle.exists()
print('Experimental, tiny AI-curated data, NOT approved for demo:',VARIANT)
files.download(str(bundle))
